## 2、短期记忆

对应《尚硅谷-09-上下文与记忆.pdf》第 2 章，PDF 第 6-38 页（到第 3 章标题之前）。

按照课件顺序整理：**内存检查点 → PostgreSQL 检查点 → 两种方式对比 → 记忆治理 → AgentState**。
沿用课件的 `agent`、`config`、`checkpointer` 和中间件命名；各节重新创建 Agent，方便分节练习。

**运行方法**：选择项目的 `langchain` Python 内核，先运行“公共准备”，再运行所需小节。
模型初始化沿用工程中百炼的 `BAILIAN_API_KEY`、`BAILIAN_API_URL` 和已有模型名。
`agent.invoke(...)` 会调用配置的真实模型；PostgreSQL 小节还需要自己的数据库。
文中的预期现象用于学习，不是本次真实 API 或数据库执行结果。


### 2.0 先理解三个组成部分

**短期记忆 = State（会话状态）+ Checkpointer（检查点保存机制）+ Thread ID（会话标识）。**

| 组成 | 作用 | 示例 |
| --- | --- | --- |
| State | 保存当前会话运行到这里的状态，默认重点是消息列表 | `state["messages"]` |
| Checkpointer | 保存、恢复运行图的状态快照 | `InMemorySaver()`、`PostgresSaver` |
| Thread ID | 确定本次调用属于哪个会话 | `config["configurable"]["thread_id"]` |

```text
传入当前新消息 + thread_id
    ↓
从同一个 checkpointer 读取该会话的历史状态
    ↓
把新消息合并进 messages → 调用模型 → 保存新的检查点
```

普通的新消息通常追加到 `messages`；同 ID 的消息可以被更新，删除则通过 `RemoveMessage` 表达。
因此通常每轮只传入新消息，不需要把所有旧消息手动再传一遍。

**“短期”描述的是会话作用域，不是保存时长。** 即使检查点存进 PostgreSQL、重启后仍在，它仍然是短期记忆；第 3 章的长期记忆通过 Store 组织跨会话数据。


### 公共准备

与现有笔记保持一致：从项目 `.env` 读取配置，再创建 `model`。
本单元只初始化客户端，不发送聊天请求；后续小节共用这里的模型。


In [1]:
import os
from typing import Any, Callable

from dotenv import load_dotenv
from rich import print as rprint
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, ToolMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override = True)

model = init_chat_model(
    model = "openai:qwen3.8-max",
    api_key = os.getenv("BAILIAN_API_KEY"),
    base_url = os.getenv("BAILIAN_API_URL"),
)


### 2.1 基于内存的持久化器

#### 2.1.1 举例 1：没有记忆

两次独立调用都只传入当前这一条消息，没有为 Agent 配置检查点。
第二次调用无法自动带上第一次的“我叫张三”。


In [2]:
agent = create_agent(
    model = model,
    tools = [],
)

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")],
})
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")],
})
print(f"Agent: {response2['messages'][-1].content}")



第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 我不知道您的名字。在我们的对话中，您还没有告诉我您的名字。如果您愿意分享，我很乐意知道！


**观察重点**：不能只凭模型碰巧答对就判断有记忆，要看第二轮的上下文是否真的包含第一轮消息。
这里 `response2["messages"]` 只包含本次用户消息和本次回复。

#### 2.1.2 举例 2：拥有记忆

**第 1 步：创建一个 `InMemorySaver`，绑定 Agent，并在两轮调用中使用同一个 `thread_id`。**


In [3]:
checkpointer = InMemorySaver()

agent = create_agent(
    model = model,
    checkpointer = checkpointer,
)

config = {"configurable": {"thread_id": "1"}}

print("\n第一轮对话：")
response1 = agent.invoke(
    {"messages": [HumanMessage("我叫张三")]},
    config = config,
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke(
    {"messages": [HumanMessage("我叫什么？")]},
    config = config,
)
print(f"Agent: {response2['messages'][-1].content}")



第一轮对话：
Agent: 哈哈，张三这个名字听起来很耳熟呀！是刚想到什么有趣的事了吗？还是想和我聊聊什么话题呢？😄

第二轮对话：
Agent: 你叫张三呀！有什么我可以帮你的吗？😊


**第 2 步：查看保存的状态。**

`agent.get_state(config)` 返回 `StateSnapshot`，消息在其 `.values["messages"]` 中。
消息是 `HumanMessage`、`AIMessage` 等对象，读取内容使用 `.content`，不是 `["content"]`。

没有工具和治理中间件时，两轮对话通常形成：`[H1, A1, H2, A2]`。
下面保留快照打印入口，但不用把课件中整段响应元数据抄进笔记。


In [4]:
latest_state = agent.get_state(config)
rprint(latest_state)

print("\n当前会话的消息：")
for msg in latest_state.values["messages"]:
    msg.pretty_print()


StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='cfb92438-dd52-4edf-962e-dd57ca947fd4'
            ),
            AIMessage(
                content='哈哈，张三这个名字听起来很耳熟呀！是刚想到什么有趣的事了吗？还是想和我聊聊什么话题呢？😄',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 102,
                        'prompt_tokens': 63,
                        'total_tokens': 165,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 71,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 102
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 63}
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.8-max',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-90233bfc-634a-98cd-a7b6-efc42a6ccd59',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a15c45-8b81-7e83-8cec-4cb2477905b2-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 63,
                    'output_tokens': 102,
                    'total_tokens': 165,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 71}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='e2a3cf06-3fae-4ed6-a4f1-4141449190dc'
            ),
            AIMessage(
                content='你叫张三呀！有什么我可以帮你的吗？😊',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 39,
                        'prompt_tokens': 108,
                        'total_tokens': 147,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 24,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 39
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 108}
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.8-max',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-175fb00f-638b-969b-8e20-ec8b885b71d5',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a15c45-9d87-7b51-83bc-53bf98757297-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 108,
                    'output_tokens': 39,
                    'total_tokens': 147,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 24}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1ccf67-45ee-6daf-8004-23751bc70bcf'
        }
    },
    metadata={'source': 'loop', 'step': 4, 'parents': {}},
    created_at='2026-10-21T02:23


当前会话的消息：
================================ Human Message =================================

我叫张三
================================== Ai Message ==================================

哈哈，张三这个名字听起来很耳熟呀！是刚想到什么有趣的事了吗？还是想和我聊聊什么话题呢？😄
================================ Human Message =================================

我叫什么？
================================== Ai Message ==================================

你叫张三呀！有什么我可以帮你的吗？😊


**第 3 步：继续同一会话，再切换会话。**

同一保存后端、同一个 `thread_id="1"` 可以继续读取张三这段历史；
换成新的 `thread_id="2"`，会使用另一份独立状态。
如果线程 2 已经在此前的练习中使用过，它会读取自己的旧历史，而不是保证为空。


In [5]:
print("\n第三轮对话（仍然是会话 1）：")
response3 = agent.invoke(
    {"messages": [HumanMessage("我刚才问了什么问题？")]},
    config = config,
)
print(f"Agent: {response3['messages'][-1].content}")

config2 = {"configurable": {"thread_id": "2"}}
print("\n切换到会话 2：")
response4 = agent.invoke(
    {"messages": [HumanMessage("你还记得我叫什么名字么？")]},
    config = config2,
)
print(f"Agent: {response4['messages'][-1].content}")



第三轮对话（仍然是会话 1）：
Agent: 你刚才问的是：**“我叫什么？”**

切换到会话 2：
Agent: 抱歉，我不记得了。因为每次对话对我来说都是全新的，我没有保存之前的记忆，所以不知道你叫什么名字。你可以再告诉我一次吗？😊


#### 2.1.3 关键步骤说明

1. `checkpointer = InMemorySaver()`：创建本次练习使用的内存保存器。
2. `create_agent(..., checkpointer=checkpointer)`：让运行图保存状态检查点。
3. `config = {"configurable": {"thread_id": "1"}}`：每次调用指定会话。

同一用户可以有写代码、写文档等不同会话；多用户也应为各自会话分配不同 ID。
`thread_id` 用来选择会话，不等同于用户身份或权限校验。

#### 2.1.4 工作原理

| 调用 | 当前传入 | 合并历史后送入模型 | 完成后保存 |
| --- | --- | --- | --- |
| 第一次 | `H1` | `[H1]` | `[H1, A1]` |
| 第二次 | `H2` | `[H1, A1, H2]` | `[H1, A1, H2, A2]` |
| 第三次 | `H3` | `[H1, A1, H2, A2, H3]` | `[H1, A1, H2, A2, H3, A3]` |

这张表描述没有工具调用、没有裁剪或删除的简单对话。检查点不只保存最终回答，运行图也会在执行步骤间保存状态。

#### 2.1.5 常见问题

| 问题 | 先检查什么 |
| --- | --- |
| Agent 不记得上一轮 | 是否绑定 `checkpointer`；是否传 `config`；`thread_id` 是否一致；是否复用了原保存器 |
| 重跑 Notebook 单元后忘记了 | `InMemorySaver()` 是否被重新执行，创建了新的空实例 |
| 重启内核后忘记了 | 内存数据随进程结束而消失，需要数据库后端才能跨进程保留 |
| 消息越来越多 | 当前状态和历史检查点都可能增长，需要按需求管理上下文与保存的数据 |
| 配了保存器但未传会话配置 | 当前版本通常会因缺少检查点配置而报错，不能当成正常的无记忆调用 |

**版本核对**：课件写“`InMemorySaver` 没有删除 API”，项目当前安装版本已提供 `checkpointer.delete_thread("1")`，用于删除该线程的全部检查点和写入。
这与换一个 `thread_id`、或只清空当前 `messages` 的含义不同；这里只说明 API，不在练习中自动执行删除。
重新创建 Agent 但复用原 `checkpointer`，也不会自动清空记忆。


### 2.2 基于外部存储介质的持久化器

#### 2.2.1 数据库环境准备

课件使用 PostgreSQL。项目的 `requirements.txt` 已包含 `langgraph-checkpoint-postgres`、`psycopg[binary]` 和 `psycopg-pool`，本节不重复安装。

运行本节前，准备好自己的数据库、用户权限和连接地址，并在项目 `.env` 中设置 `DB_URL`。
连接串形式为 `postgresql://用户名:密码@主机:端口/数据库名`；连接参数按自己的数据库要求填写。
课件中的服务器地址和口令不作为本笔记的默认配置。

#### 2.2.2 代码实现（需要 PostgreSQL，按需运行）

`PostgresSaver.from_conn_string(DB_URL)` 返回上下文管理器。
第一次使用数据库时要调用 `setup()`，它负责建表和必要迁移；再次调用不会清空已有聊天历史。
在 `with` 块内完成依赖此连接的 Agent 操作，退出后连接会关闭。


In [11]:
from langgraph.checkpoint.postgres import PostgresSaver

load_dotenv(override = True)
DB_URL = os.environ["DB_URL"]

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()

    agent = create_agent(
        model = model,
        checkpointer = checkpointer,
    )
    config = {"configurable": {"thread_id": "1"}}

    response1 = agent.invoke(
        {"messages": [HumanMessage("你好，我是老王")]},
        config = config,
    )
    print("=" * 30, "-> 第一次调用 <-", "=" * 30)
    for msg in response1["messages"]:
        msg.pretty_print()

    response2 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁？")]},
        config = config,
    )
    print("=" * 30, "-> 第二次调用 <-", "=" * 30)
    for msg in response2["messages"]:
        msg.pretty_print()


============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。今天有什么我可以帮你的吗？
============================== -> 第二次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。今天有什么我可以帮你的吗？
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你是老王呀！我们刚刚才打过招呼的。有什么想聊聊或者需要帮忙的吗？


预期第二轮能从该线程历史中找到“老王”。再次运行本单元会继续同一数据库中线程 `1` 的历史，并追加新消息。
数据库后端和内存后端是不同的保存空间，相同的字符串 `"1"` 不会让二者自动共享数据。

#### 2.2.3 查看持久化数据

在 `psql` 中执行以下命令，而不是直接放进 Python 代码单元：

```sql
\l
\dn
SELECT current_schema();
\dt
```

依次查看数据库、Schema、当前 Schema、当前 Schema 下的表。
PostgreSQL 的层次是 **Database → Schema（命名空间）→ Table**，Schema 不等于表分区。

| 表 | 保存什么 |
| --- | --- |
| `checkpoints` | 每个线程在各执行时刻的检查点信息 |
| `checkpoint_blobs` | 单独存储的通道值数据 |
| `checkpoint_writes` | 执行任务产生的中间写入、待处理写入 |
| `checkpoint_migrations` | 检查点表结构的迁移版本 |

可用只读 SQL 查看线程对应的检查点：

```sql
SELECT thread_id, checkpoint_ns, checkpoint_id, parent_checkpoint_id
FROM checkpoints
WHERE thread_id = '1'
ORDER BY checkpoint_id DESC
LIMIT 10;
```


### 2.3 对比两种方式

| 对比项 | `InMemorySaver` | `PostgresSaver` |
| --- | --- | --- |
| 保存位置 | 当前 Python 进程的内存 | 指定 PostgreSQL 数据库 |
| 同一实例、相同会话 ID | 继续历史 | 继续历史 |
| 重建 Saver | 新实例为空；只有 ID 相同不够 | 指向同一数据库、同一检查点命名空间和线程时可恢复 |
| 重启 Python 或 Notebook 内核 | 原内存历史丢失 | 数据库中的历史仍可读取 |
| 多进程 | 不自动共享各自内存中的数据 | 可以连接同一数据库保存、读取状态 |
| 适用场景 | 学习、测试、调试 | 需要跨进程保留状态的应用 |

课件的对比实验：连续进行“我是谁？→ 我是老王 → 我是谁？”三轮对话，再重新运行整段创建 Saver 的代码。
内存版每次重新创建 `InMemorySaver()` 都从空历史开始；数据库版重新连接同一后端和线程，历史会继续累积。

**Notebook 中要分清两种操作**：只重跑 `agent.invoke(...)` 可以复用原内存；连同 `InMemorySaver()` 一起重跑则会创建新的保存器。


### 2.4 记忆治理策略（上下文管理）

历史不断增长会增加模型输入、调用成本，也可能超出上下文窗口或让无关旧内容影响回答。

| 策略 | 处理方式 | 主要取舍 |
| --- | --- | --- |
| 消息裁剪 | 保留首条或最近部分消息 | 简单，但可能丢失中间事实 |
| 消息删除 | 按消息 ID 从当前状态移除 | 明确清理，但其他消息仍可能提及同一事实 |
| 摘要 | 旧历史压缩成摘要，再保留最近消息 | 尽量保留含义，但有摘要调用成本与信息损失 |
| 自定义过滤 | 按业务规则选择消息 | 需要维护消息顺序及工具调用关系 |

**区分模型输入与持久状态**：仅在 `wrap_model_call` 中改写请求，可只影响本次模型输入；
本节课件的裁剪在 `before_model` 中返回状态更新，因此也会改变后续检查点里的当前 `messages`。
裁剪或删除当前状态，不代表保存后端中的所有旧检查点都被物理清除。


#### 2.4.1 消息裁剪

沿用课件的规则：消息不超过 3 条时不处理；超过后保留第一条，再按总数奇偶取最近 3 或 4 条。
通过先返回 `RemoveMessage(id=REMOVE_ALL_MESSAGES)`，再放入保留消息，实现替换当前消息列表。
仅返回一个较短的列表，默认消息 Reducer 不会把它理解为“删除没出现的旧消息”。

**本例的限制**：第一条是用户消息“你好，我是老王”，不是系统提示；这个按奇偶取消息的规则只用于课件的简单对话演示。
它不能直接套到任意工具对话，工具调用与对应 `ToolMessage` 必须保持配对。


In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]
    if len(messages) <= 3:
        return None

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id = REMOVE_ALL_MESSAGES),
            *new_messages,
        ],
    }


In [ ]:
agent = create_agent(
    model = model,
    middleware = [trim_messages],
    checkpointer = InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke(
    {"messages": [HumanMessage("告诉我，你是谁？我是谁？")]},
    config,
)

for msg in final_response["messages"]:
    msg.pretty_print()


用 `H` 表示用户消息，`A` 表示模型回复：

| 第几轮 | 调用模型前的消息 | 裁剪后真正保留的消息 |
| --- | --- | --- |
| 1 | `[H1]` | `[H1]` |
| 2 | `[H1, A1, H2]` | `[H1, A1, H2]` |
| 3 | `[H1, A1, H2, A2, H3]` | `[H1, A1, H2, A2, H3]`，本次没有减少 |
| 4 | `[H1, A1, H2, A2, H3, A3, H4]` | `[H1, A2, H3, A3, H4]` |

第四轮里，“从现在起，你叫小王”的用户消息 `H2` 已被裁掉，首条 `H1` 仍在。
但 `A2` 可能仍提到“小王”，所以不能保证模型一定忘记这个名字。应结合消息列表检查，不能把课件某一次输出当作固定结果。


#### 2.4.2 消息删除

沿用课件：在 `after_model` 中，消息超过 5 条就删除最旧的超出部分。
`RemoveMessage(id=m.id)` 是给 `add_messages` Reducer 的删除指令；被删除消息不会继续留在合并后的当前消息列表。
课件用“墓碑一直追加在当前消息池、读取时再过滤”来解释，这不准确；旧检查点保留历史是另一层机制。


In [ ]:
from langchain.agents.middleware import after_model


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    if len(messages) > 5:
        to_delete = len(messages) - 5
        return {
            "messages": [RemoveMessage(id = m.id) for m in messages[:to_delete]],
        }
    return None


In [ ]:
agent = create_agent(
    model = model,
    middleware = [delete_old_messages],
    checkpointer = InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke(
    {"messages": [HumanMessage("告诉我，你是谁？我是谁？")]},
    config,
)

for msg in final_response["messages"]:
    msg.pretty_print()


| 轮次 | 本轮回复后、删除前的条数 | 删除条数 | 删除后的条数 |
| --- | --- | --- | --- |
| 1 | 2 | 0 | 2 |
| 2 | 4 | 0 | 4 |
| 3 | 6 | 1 | 5 |
| 4 | 7 | 2 | 5 |

**计数更正**：第三轮已经删到 5 条，第四轮再加一条用户消息和一条回复，是 7 条，不是课件分析中的 8 条。

删除“你好，我是老王”后，保留的 AI 回复中可能还有“老王”，所以事实仍可能被推断出来。
该演示按条数保留，可能以 AI 消息开头；部分模型对消息起始角色和工具配对有要求，实际应用要按合法的对话边界处理。


#### 2.4.3 摘要

使用内置 `SummarizationMiddleware`：把较早消息压缩成摘要，并保留最近部分消息原文。
课件用 `model_out` 负责回答、`model_in` 负责摘要；这里先共用公共准备的 `model`，也可把 `model_in` 换为自己配置的摘要模型。

| 参数 | 本例设置 | 含义 |
| --- | --- | --- |
| `model` | `model_in` | 执行摘要的模型 |
| `trigger` | `[("tokens", 100)]` | 达到 token 触发条件时尝试摘要 |
| `keep` | `("messages", 2)` | 按消息条数保留最近部分，不是两轮对话 |
| `summary_prompt` | 含 `{messages}` | 把待摘要历史插入模板 |

100 是为了演示设置的很小阈值，不是模型窗口大小，也不保证摘要后的全部输入不超过 100 tokens。
默认 token 计数是估算；真正执行摘要还需要存在可压缩的旧消息。
第一次只有一条很长的消息时，可能因保留规则而不摘要；后续累积出可压缩的历史后再触发。


In [12]:
from langchain.agents.middleware import SummarizationMiddleware

model_out = model
model_in = model

agent = create_agent(
    model = model_out,
    tools = [],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model_in,
            trigger = [("tokens", 100)],
            keep = ("messages", 2),
            summary_prompt = "对历史消息摘要，保留姓名、职业及关键事实。消息列表如下\n{messages}",
        ),
    ],
)
config = {"configurable": {"thread_id": "1"}}

conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20,
    "请总结一下我的信息",
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [HumanMessage(msg)]},
        config = config,
    )
    for message in response["messages"]:
        message.pretty_print()
    print("*" * 50)


================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

张三你好！很高兴认识你。作为工程师，如果你在工作中遇到任何技术问题，或者需要讨论解决方案、寻找灵感，随时都可以和我交流。有什么我可以帮你的吗？😊
**************************************************
================================ Human Message =================================

Here is a summary of the conversation to date:

张三，工程师。
=================================

In [13]:
final_state = agent.get_state(config)
print("\n最终保存在 State 中的消息：")
for msg in final_state.values.get("messages", []):
    msg.pretty_print()



最终保存在 State 中的消息：
================================ Human Message =================================

Here is a summary of the conversation to date:

张三，工程师。
================================== Ai Message ==================================

张三你好！很高兴认识你。作为工程师，如果你在工作中遇到任何技术问题，或者需要讨论解决方案、寻找灵感，随时都可以和我交流。有什么我可以帮你的吗？😊
================================ Human Message =================================

请总结一下我的信息
================================== Ai Message ==================================

根据您提供的信息，您的资料如下：

- **姓名**：张三  
- **职业**：工程师  

如果您还有其他信息需要补充或更新，随时告诉我！


```text
历史消息达到触发条件，且有可摘要的早期历史
    ↓
摘要模型压缩早期消息
    ↓
新 messages = 摘要消息 + 最近保留的消息
    ↓
回答模型继续处理当前问题
```

摘要有可能遗漏关键细节；姓名、约束、关键决定要通过实际结果核验，不能认为必然保留。
生产场景应按真实模型窗口、系统提示、工具定义和预期输出预留空间，再调整阈值及摘要频率。
摘要会产生额外模型请求，是否节省总成本取决于历史长度、触发频率及所选模型。

#### 2.4.4 自定义过滤策略

课件未再提供独立代码：通过中间件按业务规则筛选消息即可。
需要先确定是只修改当前模型请求，还是更新持久状态；同时保留当前问题，并维护工具调用和结果的对应关系。


### 2.5 了解：State 的理解

`AgentState` 是 `TypedDict`，用于描述运行图的状态结构；运行时按字典方式访问。

| 字段 | 含义 | 读取方式 |
| --- | --- | --- |
| `messages` | 运行到当前节点的消息列表，使用 `add_messages` 合并 | `state["messages"]` |
| `jump_to` | 中间件指定的临时跳转目标 | `state.get("jump_to")` |
| `structured_response` | 启用结构化输出后生成的结构化结果 | `state.get("structured_response")` |

`Required` 表示键必需；`NotRequired` 表示键可以不出现，不等于“值可以为 None”。
`jump_to` 的类型另外包含 `None`，并且它是临时控制信息，不是用户的长期资料。

下面保留课件的天气案例：

```text
HumanMessage（查询北京天气）
    ↓ before_model
人工构造 AIMessage(tool_calls) + jump_to="tools"
    ↓ tools
get_weather → ToolMessage
    ↓ model
生成 WeatherInfo → after_agent 观察最终 State
```

天气返回固定教学模拟数据；只在最后的模型生成阶段调用真实模型。


In [20]:
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.tools.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call, after_agent,before_model,AgentState
from langgraph.runtime import Runtime
from langgraph.types import Command


class WeatherInfo(BaseModel):
    """城市天气情况。"""

    city: str = Field(description = "城市名称")
    temperature: str = Field(description = "气温")
    desc: str = Field(description = "当日天气概述")


@tool(parse_docstring = True)
def get_weather(city: str) -> str:
    """获取当日天气，返回教学模拟数据。

    Args:
        city: 城市名称。
    """
    return f"[教学模拟] [{city}] 今天气温9~16度，万里无云，天气不错适合外出"


In [21]:
@before_model(can_jump_to = ["tools"])
def direct_tool_call(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_msg = state["messages"][-1]
    if isinstance(last_msg, HumanMessage) and "天气" in last_msg.text and "北京" in last_msg.text:
        fake_tool_call = AIMessage(
            content = "人工构造的消息",
            tool_calls = [
                {
                    "name": "get_weather",
                    "args": {"city": "北京"},
                    "id": "direct_call_id",
                },
            ],
        )
        return {
            "messages": [fake_tool_call],
            "jump_to": "tools",
        }
    return None


@wrap_tool_call
def first_check(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    print("=" * 30, "-> In first_check Middleware <-", "=" * 30)
    print(f"jump_to = {request.state.get('jump_to')}")
    print(f"structured_response = {request.state.get('structured_response')}")
    return handler(request)


@after_agent
def final_check(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("=" * 30, "-> In final_check Middleware <-", "=" * 30)
    for msg in state["messages"]:
        msg.pretty_print()
    print(f"jump_to = {state.get('jump_to')}")
    print(f"structured_response = {state.get('structured_response')}")
    return None


In [22]:
agent = create_agent(
    model = model,
    response_format = WeatherInfo,
    middleware = [direct_tool_call, first_check, final_check],
    tools = [get_weather],
)

response = agent.invoke({
    "messages": [HumanMessage("请帮我查询北京当日天气")],
})

rprint(response["structured_response"])


============================== -> In first_check Middleware <- ==============================
jump_to = tools
structured_response = None
============================== -> In final_check Middleware <- ==============================
================================ Human Message =================================

请帮我查询北京当日天气
================================== Ai Message ==================================

人工构造的消息
Tool Calls:
  get_weather (direct_call_id)
 Call ID: direct_call_id
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

[教学模拟] [北京] 今天气温9~16度，万里无云，天气不错适合外出
================================== Ai Message ==================================
Tool Calls:
  WeatherInfo (call_225d30a8fb3248c8bb44f435)
 Call ID: call_225d30a8fb3248c8bb44f435
  Args:
    city: 北京
    temperature: 9~16℃
    desc: 万里无云，天气不错，适合外出
================================= Tool Message =================================
Name: WeatherInfo

Returning st

WeatherInfo(city='北京', temperature='9~16℃', desc='万里无云，天气不错，适合外出')

**观察重点**：

1. `direct_tool_call` 在首次调用模型前，添加一条合法的工具请求，并通过 `jump_to="tools"` 进入工具节点。
2. `first_check` 观察工具调用时的状态；此时通常还没有最终 `structured_response`。
3. 工具返回后仍需模型整理结果；`after_agent` 中观察到的 `structured_response` 应为 `WeatherInfo` 对象。
4. `jump_to` 是临时控制值，结束时读取通常为 `None`；不能假定每个节点都保存着这个键。

本例没有传入 `checkpointer`：运行期间仍然存在 State，但不会自动把这次状态带到下一次独立调用。
`response_format=WeatherInfo` 的策略选择依赖模型能力与适配器；如果所用兼容接口需要工具式结构化输出，可显式使用 `ToolStrategy(WeatherInfo)`（从 `langchain.agents.structured_output` 导入）。

### 本章回顾

- 会话能否连续，先看保存后端、`thread_id` 和 Saver 实例是否匹配。
- 内存版适合练习；PostgreSQL 能在重新连接后恢复已保存的会话状态。
- 管理上下文要区分本次模型输入、当前 State，以及保存后端里的旧检查点。
- `messages` 是消息对象列表，`structured_response` 是结构化结果，`jump_to` 是临时流程控制。

**来源与核对**：主体来自课件第 2 章；删除 API、Reducer、摘要保留规则和 `AgentState` 说明，按项目当前安装的 `langchain==1.2.12`、`langchain-core==1.2.18`、`langgraph==1.1.2`、`langgraph-checkpoint==4.0.1`、`langgraph-checkpoint-postgres==3.0.5` 的接口与源码核对。未升级依赖。

官方延伸阅读：`https://docs.langchain.com/oss/python/langchain/short-term-memory`；`https://docs.langchain.com/oss/python/langgraph/persistence`。
